In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [ ]:
ds = pd.read_csv("https://raw.githubusercontent.com/datasets/breast-cancer/refs/heads/main/data/breast-cancer.csv")
ds.head()

,age,mefalsepause,tumor-size,inv-falsedes,falsede-caps,deg-malig,breast,breast-quad,irradiat,class
0,40-49,premefalse,15-19,0-2,True,3,right,left_up,False,recurrence-events
1,50-59,ge40,15-19,0-2,False,1,right,central,False,false-recurrence-events
2,50-59,ge40,35-39,0-2,False,2,left,left_low,False,recurrence-events
3,40-49,premefalse,35-39,0-2,True,3,right,left_low,True,false-recurrence-events
4,40-49,premefalse,30-34,3-5,True,2,left,right_up,False,recurrence-events


In [ ]:
X = pd.get_dummies(ds.iloc[:, :-1],drop_first=True)
le = LabelEncoder()
y = le.fit_transform(ds.iloc[:, -1])

In [ ]:
X_train, X_test, y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [ ]:
#Standard scaling
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [ ]:
#CONVERTING NUMPY ARRAYS TO PYTORCH TENSORS
import torch
X_train = torch.from_numpy(X_train)
X_test = torch.from_numpy(X_test)
y_train = torch.from_numpy(y_train)
y_test = torch.from_numpy(y_test)

In [ ]:
#DEFINING A MODEL

class mysimpleNN() :
  def __init__(self,X) :
    self.weights = torch.rand(X.shape[1],1,dtype = torch.float64,requires_grad=True) #here we writing (X.shape[1],1)--> for defining shape of rand function like 30 rows and 1 column

    self.bias = torch.zeros(1,dtype = torch.float64,requires_grad=True)

  def forward(self,X) :
      z = torch.matmul(X,self.weights) + self.bias
      y_pred = torch.sigmoid(z)
      return y_pred



  def loss_function(self, y_pred, y):
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1 - epsilon)

    # Binary Cross Entropy Loss
    loss = -(y * torch.log(y_pred) + (1 - y) * torch.log(1 - y_pred)).mean()

    return loss


In [ ]:
#LEARNING PARAMETERS
learning_rate = 0.1
epochs = 25

In [ ]:
# TRAINING PIPELINE

#CREATING MODEL
model = mysimpleNN(X_train)

#DEFINE LOOP
for epoch in range(epochs) :

  #forward
  y_pred = model.forward(X_train)

  #loss
  loss = model.loss_function(y_pred,y_train)



  #backward
  loss.backward()

  #PARAMETERS UPDATING
  with torch.no_grad() :
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad


  #RESET GRADIENTS
  model.weights.grad.zero_()
  model.bias.grad.zero_()

  print(f'Epoch:{epoch+1} , Loss:{loss}')



Epoch:1 , Loss:1.3914245205182953
Epoch:2 , Loss:1.3676735455163882
Epoch:3 , Loss:1.3443478223705074
Epoch:4 , Loss:1.3214539130372964
Epoch:5 , Loss:1.2989984056700359
Epoch:6 , Loss:1.2769878972614368
Epoch:7 , Loss:1.2554289669058907
Epoch:8 , Loss:1.2343281393049947
Epoch:9 , Loss:1.2136918384925146
Epoch:10 , Loss:1.1935263321162337
Epoch:11 , Loss:1.1738376669689348
Epoch:12 , Loss:1.1546315967978176
Epoch:13 , Loss:1.1359135037288715
Epoch:14 , Loss:1.1176883149140262
Epoch:15 , Loss:1.0999604162290608
Epoch:16 , Loss:1.0827335650156507
Epoch:17 , Loss:1.066010803963422
Epoch:18 , Loss:1.0497943782635566
Epoch:19 , Loss:1.0340856581351137
Epoch:20 , Loss:1.0188850687302702
Epoch:21 , Loss:1.0041920292701574
Epoch:22 , Loss:0.9900049030574198
Epoch:23 , Loss:0.9763209597608425
Epoch:24 , Loss:0.9631363510814724
Epoch:25 , Loss:0.9504461005955467


In [ ]:
model.bias

tensor([-0.3627], dtype=torch.float64, requires_grad=True)

In [ ]:
#MODEL EVALUATION
with torch.no_grad() :
  y_pred = model.forward(X_test)
  y_pred = (y_pred>0.7).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy:{accuracy}')

Accuracy:0.6294214725494385
